In [8]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import least_squares

import equations as eq

RESULTS = Path("results")
env = json.loads((RESULTS / "env.json").read_text())
data = pd.read_csv(RESULTS / "measurements.csv")

eq.BW_PEAK = env["bw_peak"]          # паспорт той карты, на которой мерили
eq.FLOPS_PEAK = env["flops_peak"]

ok = data[~data["oom"]]
train = ok[~ok["is_validation"]]
val = ok[ok["is_validation"]]


def error(predicted, measured):
    """Средняя относительная ошибка, %."""
    return float(np.abs(predicted / measured - 1).mean() * 100)


print(f"{len(data)} точек, из них OOM {data['oom'].sum()}")
print(f"обучение {len(train)}, валидация {len(val)}")

132 точек, из них OOM 0
обучение 63, валидация 69


In [9]:
for name, df in ("обучение", train), ("валидация", val):
    predicted = eq.memory(df["S"].values, df["B"].values)
    print(f"memory, {name}: ошибка {error(predicted, df['memory'].values):.0f}%")


# Латентность: theta0, eta_mem, eta_compute
# Сверху eta единицей не ограничиваем — выход за неё это диагностика, а не ошибка:
# eta_compute > 1 означает, что cuDNN выполняет меньше операций, чем мы насчитали,
# eta_mem > 1 — что часть трафика осела в кэше и до DRAM не доехала.
NAMES = ("theta0", "eta_mem", "eta_compute")


def latency_residuals(params, df):
    # Расчёт логарифма ошибки между предсказанными значениями и фактическими
    theta = dict(zip(NAMES, params))
    predicted = eq.latency(df["S"].values, df["B"].values, theta)
    return np.log(predicted) - np.log(df["latency"].values)


# Подгоняем параметры:
# - theta0 (базовое время)
# - eta_mem (эффективность производительности памяти)
# - eta_compute (эффекивность производительности арифметики)
fit = least_squares(
    latency_residuals,                             # Как считаем ошибку
    x0=[train["latency"].min(), 0.6, 0.3],         # Начальные значения параметров
    bounds=([1e-7, 1e-3, 1e-3], [1e-1, 3.0, 3.0]), # В каких диапазонах ищем
    args=(train,),
)

theta = dict(zip(NAMES, fit.x))

print(f"theta0      {theta['theta0'] * 1e6:6.0f} мкс")
print(f"eta_mem     {theta['eta_mem']:6.2f}  = {theta['eta_mem'] * eq.BW_PEAK / 1e9:5.0f} ГБ/с")
print(f"eta_compute {theta['eta_compute']:6.2f}  = {theta['eta_compute'] * eq.FLOPS_PEAK / 1e12:5.2f} ТФЛОП/с")

# Считаем ошибки после подгонки
for name, df in ("обучение", train), ("валидация", val):
    predicted = eq.latency(df["S"].values, df["B"].values, theta)
    print(f"latency, {name}: ошибка {error(predicted, df['latency'].values):.0f}%")

# Какой член выигрывает max в каждой точке. Если memory не побеждает нигде, то
# eta_mem на невязку не влияла и осталась равной начальному приближению —
# это не откалиброванное значение, и в README так и надо написать.
times = [
    np.full(len(ok), theta["theta0"]),
    eq.bytes_moved(ok["S"].values, ok["B"].values) / (theta["eta_mem"] * eq.BW_PEAK),
    eq.flops(ok["S"].values, ok["B"].values) / (theta["eta_compute"] * eq.FLOPS_PEAK),
]
regimes = np.array(["launch", "memory", "compute"])[np.argmax(times, axis=0)]
print("режимы:", dict(zip(*np.unique(regimes, return_counts=True))))

memory, обучение: ошибка 28%
memory, валидация: ошибка 19%
theta0         547 мкс
eta_mem       0.60  =   192 ГБ/с
eta_compute   0.36  =  2.91 ТФЛОП/с
latency, обучение: ошибка 19%
latency, валидация: ошибка 21%
режимы: {np.str_('compute'): np.int64(109), np.str_('launch'): np.int64(23)}


In [10]:
# Подгоняем параметры для энерегии
assert data["energy"].notna().any(), "энергия не измерялась, NVML был недоступен"
P_IDLE = env["p_idle"]


def energy_residuals(params, df):
    # Функция расчёта логарифма ошибки между предсказанными и фактическими значениями
    theta_energy = {**theta, "p_idle": P_IDLE, "e_flop": params[0], "e_byte": params[1]}
    predicted = eq.energy(df["S"].values, df["B"].values, theta_energy)
    return np.log(predicted) - np.log(df["energy"].values)


# Верхние границы из бюджета мощности: динамическая часть не может стоить больше,
# чем TDP минус простой. Делить бюджет надо на РЕАЛЬНО достигнутые скорости, а не
# на паспортные: карта не выходит на 8.1 ТФЛОП/с, поэтому граница из пикового
# значения оказывается в разы жёстче физической, и фит упирается в неё.
budget = env["tdp"] - P_IDLE
peak_flops = (eq.flops(ok["S"].values, ok["B"].values) / ok["latency"].values).max()
peak_bytes = (eq.bytes_moved(ok["S"].values, ok["B"].values) / ok["latency"].values).max()

# Подгоняем параметры по энергии
fit_energy = least_squares(
    energy_residuals,
    x0=[3e-12, 5e-11],
    bounds=([0.0, 0.0], [budget / peak_flops, budget / peak_bytes]),
    args=(train,),
)
theta_energy = {"p_idle": P_IDLE, "e_flop": fit_energy.x[0], "e_byte": fit_energy.x[1]}

print(f"p_idle {P_IDLE:.1f} Вт (измерено, не подбиралось)")
print(f"e_flop {theta_energy['e_flop'] * 1e12:.2f} пДж/FLOP")
print(f"e_byte {theta_energy['e_byte'] * 1e12:.1f} пДж/байт")
print(f"отношение e_byte/e_flop: {theta_energy['e_byte'] / theta_energy['e_flop']:.0f}")

full = {**theta, **theta_energy}
for name, df in ("обучение", train), ("валидация", val):
    predicted = eq.energy(df["S"].values, df["B"].values, full)
    print(f"energy, {name}: ошибка {error(predicted, df['energy'].values):.0f}%")

power = eq.energy(ok["S"].values, ok["B"].values, full) / eq.latency(ok["S"].values, ok["B"].values, theta)
print(f"предсказанная мощность {power.min():.0f}..{power.max():.0f} Вт при TDP {env['tdp']:.0f}")

p_idle 29.8 Вт (измерено, не подбиралось)
e_flop 3.11 пДж/FLOP
e_byte 881.1 пДж/байт
отношение e_byte/e_flop: 283
energy, обучение: ошибка 20%
energy, валидация: ошибка 21%
предсказанная мощность 37..74 Вт при TDP 70


In [6]:
# Энергия: e_flop и e_byte при измеренной P_idle и фиксированной theta
assert data["energy"].notna().any(), "энергия не измерялась, NVML был недоступен"
P_IDLE = env["p_idle"]


def energy_residuals(params, df):
    theta_energy = {**theta, "p_idle": P_IDLE, "e_flop": params[0], "e_byte": params[1]}
    predicted = eq.energy(df["S"].values, df["B"].values, theta_energy)
    return np.log(predicted) - np.log(df["energy"].values)


# Верхние границы из бюджета мощности: динамическая часть не может стоить больше,
# чем TDP минус простой, даже если бы весь бюджет ушёл в один ресурс.
budget = env["tdp"] - P_IDLE
fit_energy = least_squares(
    energy_residuals,
    x0=[1e-12, 5e-11],
    bounds=([0.0, 0.0], [budget / eq.FLOPS_PEAK, budget / eq.BW_PEAK]),
    args=(train,),
)
theta_energy = {"p_idle": P_IDLE, "e_flop": fit_energy.x[0], "e_byte": fit_energy.x[1]}

print(f"p_idle {P_IDLE:.1f} Вт (измерено, не подбиралось)")
print(f"e_flop {theta_energy['e_flop'] * 1e12:.2f} пДж/FLOP")
print(f"e_byte {theta_energy['e_byte'] * 1e12:.1f} пДж/байт")
print(f"отношение e_byte/e_flop: {theta_energy['e_byte'] / theta_energy['e_flop']:.0f}")

full = {**theta, **theta_energy}
for name, df in ("обучение", train), ("валидация", val):
    predicted = eq.energy(df["S"].values, df["B"].values, full)
    print(f"energy, {name}: ошибка {error(predicted, df['energy'].values):.0f}%")

power = eq.energy(ok["S"].values, ok["B"].values, full) / eq.latency(ok["S"].values, ok["B"].values, theta)
print(f"предсказанная мощность {power.min():.0f}..{power.max():.0f} Вт при TDP {env['tdp']:.0f}")


p_idle 29.8 Вт (измерено, не подбиралось)
e_flop 4.97 пДж/FLOP
e_byte 125.7 пДж/байт
отношение e_byte/e_flop: 25
energy, обучение: ошибка 28%
energy, валидация: ошибка 36%
предсказанная мощность 31..49 Вт при TDP 70


In [11]:
(RESULTS / "theta.json").write_text(json.dumps({
    "env": env,
    "theta": theta,
    "theta_energy": theta_energy,
}, indent=2, ensure_ascii=False))
print("записано в", RESULTS / "theta.json")


записано в results/theta.json
